In [ ]:
# 충돌 원인 제거
!pip -q uninstall -y torchaudio

# 깔끔히 맞는 조합 설치 (A100/코랩에서 안전한 cu121 빌드)
!pip -q install --upgrade pip
!pip -q install "torch==2.3.1+cu121" "torchvision==0.18.1+cu121" --index-url https://download.pytorch.org/whl/cu121

# 필수 라이브러리
!pip -q install "transformers>=4.43.3" "accelerate>=0.33.0" "peft>=0.12.0" "datasets" "einops" "pillow"
!pip -q install "bitsandbytes==0.43.3"
!pip -q install qwen-vl-utils  # Qwen2-VL 쓸 경우
!pip -q install --upgrade "transformers>=4.45.2" "accelerate>=0.34.2" "qwen-vl-utils>=0.0.8"
!pip install -q triton
!pip install -q bitsandbytes
!pip -q install "triton==2.2.0"
from google.colab import drive, files
drive.mount('/content/drive')
!pip -q install -U transformers accelerate peft pillow einops
!pip -q install -U bitsandbytes  # 설치 실패하면 4bit 경로 사용 불가 → 아래 코드에서 USE_4BIT=False 유지

# 1) 기존 PyTorch & CUDA 파편 패키지 제거
!pip -q uninstall -y torch torchvision torchaudio triton xformers \
  nvidia-cublas-cu12 nvidia-cudnn-cu12 nvidia-cuda-nvrtc-cu12 \
  nvidia-cuda-runtime-cu12 nvidia-cuda-cupti-cu12 nvidia-cusparse-cu12

# 2) PyTorch 2.3.1 + cu121 설치 (토치/비전/오디오 버전 꼭 맞추기)
!pip -q install --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1

# 3) 나머지 라이브러리(bnb는 안 써도 됨; 쓸거면 추가로 설치)
!pip -q install -U "transformers==4.44.2" "accelerate>=0.33.0" "peft>=0.11.1" pillow einops

# (선택) bitsandbytes 쓰고 싶으면 이 줄 추가 (4bit 쓸 때만)
# !pip -q install -U bitsandbytes
import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)
# 기대: torch: 2.3.1+cu121 | CUDA: 12.1

!pip -q install -U "transformers>=4.46.0" "accelerate>=0.33.0" "peft>=0.11.1" pillow einops



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.8 MB/s eta 0:00:00
Mounted at /content/drive
torch: 2.3.1+cu121 | CUDA: 12.1


In [ ]:
# =========================================================
# Qwen2-VL-7B + LoRA(4bit) OOM-safe 학습/저장/재개/제출 스크립트
# (Colab 24h 대응, 8k 서브셋 옵션, Torch<2.6 CVE 우회, 끊김 방지 오토그로우 추론)
# - PATCH①: HTTP 세션/재시도/UA → URL 이미지 로딩 성공률↑
# - PATCH②: 프롬프트 보존, '정답'만 컷 → 라벨/프롬프트 정렬 깨짐 방지
# - PATCH③: 추론시 반복 억제 + how-many 숫자만 추출
# - PATCH④: 오토그로우(min_new_tokens + 부족하면 이어 생성) → "중간에 끊김" 방지
# =========================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, io, time, random, math, gc, hashlib, warnings, logging, re, inspect
import torch, pandas as pd, numpy as np
from PIL import Image, ImageFile
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry  # ← 올바른 위치
from torch.utils.data import Dataset as TorchDataset
from huggingface_hub import login
from safetensors.torch import load_file as safe_load_file

# ---- transformers v5 경고 호환 ----
try:
    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq
except Exception:
    from transformers import AutoModelForVision2Seq

from transformers import (
    AutoProcessor,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    TrainerCallback
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ---- quiet some noisy warnings (optional) ----
os.environ.setdefault("TRANSFORMERS_VERBOSITY", "error")
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module="torch.utils.checkpoint")
warnings.filterwarnings("ignore", message=".*AutoModelForVision2Seq.*")
Image.MAX_IMAGE_PIXELS = 300_000_000
ImageFile.LOAD_TRUNCATED_IMAGES = True

GLOBAL_START = time.time()  # 전체 런 시작 시각

# -----------------------------
# 🧷 사용자 조정 지점
# -----------------------------
MODEL_ID    = "Qwen/Qwen2-VL-7B-Instruct"
CSV_TRAIN   = "/content/drive/MyDrive/multimodal_challenge_aju/data/deep_chal_multitask_dataset.parquet"
CSV_VAL     = "/content/drive/MyDrive/multimodal_challenge_aju/data/deep_chal_multitask_dataset_sample.parquet"
CSV_TEST    = "/content/drive/MyDrive/multimodal_challenge_aju/data/deep_chal_multitask_dataset_test.parquet"

SAVE_DIR    = "/content/drive/MyDrive/multimodal_challenge_aju/outputs_qwen2vl_lora_total_last_last4"
CKPT_DIR    = os.path.join(SAVE_DIR, "checkpoints")
FINAL_DIR   = SAVE_DIR
SUBMIT_OUT  = os.path.join(SAVE_DIR, "submission.csv")
SUBMIT_PART = SUBMIT_OUT + ".partial"
IMG_CACHE_DIR = os.path.join(SAVE_DIR, "img_cache")
SUBSET_IDX_PATH = os.path.join(SAVE_DIR, "train_subset_indices.csv")  # 8k 인덱스 저장

# 새 학습 강제 시작(이전 LoRA/체크포인트 무시)
FORCE_FRESH = False

# ---- 부분 학습 모드 (빠른 러프런) ----
USE_TRAIN_SUBSET = False
SUBSET_SIZE      = 8000
SUBSET_SEED      = 42
SUBSET_GROUP_CANDIDATES = ["input_type", "task", "lang"]  # 있으면 층화

# 시간/저장 설정 (24h 대비)
LIMIT_HOURS          = 23
AUTOSAVE_EVERY_MIN   = 30
INFER_SAVE_EVERY     = 200

# 메모리/학습 하이퍼
MAX_TEXT_TOKENS = 1024
MAX_IMAGE_EDGE  = 448          # OOM 시 384/320로 낮추기
LORA_R          = 32           # OOM 시 16
GRAD_ACC_STEPS  = 16
PER_DEV_BATCH   = 1            # 안전 기본값(가변 해상도)

# 토큰
token = os.getenv("HF_TOKEN")
if token:
    login(token=token, add_to_git_credential=True)
    os.environ["HUGGINGFACE_HUB_TOKEN"] = token

# 재현성
torch.manual_seed(42); random.seed(42); np.random.seed(42)
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda)

# 성능/안정화
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
USE_BF16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
USE_FP16 = not USE_BF16 and torch.cuda.is_available()

# -----------------------------
# 유틸
# -----------------------------
def safe_read_df(path):
    try:
        if path.endswith(".parquet"):
            return pd.read_parquet(path)
        if path.endswith(".csv"):
            return pd.read_csv(path)
        try:  # 확장자 불명 시 parquet→csv 순으로 시도
            return pd.read_parquet(path)
        except Exception:
            return pd.read_csv(path)
    except Exception as e:
        raise RuntimeError(f"파일 읽기 실패: {path} | {e}")

def build_text_prompt(context, question):
    parts = []
    if context and str(context).strip():
        parts.append(str(context).strip())
    if question and str(question).strip():
        parts.append("Question: " + str(question).strip())
    return "\n".join(parts).strip() if parts else ""

def _round_to_multiple(x, m):
    x = max(m, int(round(x / m)) * m)
    return x

def _cache_path(url: str, max_edge: int):
    os.makedirs(IMG_CACHE_DIR, exist_ok=True)
    h = hashlib.sha1(f"{url}|{max_edge}".encode()).hexdigest()
    return os.path.join(IMG_CACHE_DIR, f"{h}.jpg")

# ---------- PATCH①: 견고한 HTTP 세션(재시도/UA) ----------
def _build_requests_session():
    s = requests.Session()
    retries = Retry(
        total=3, connect=3, read=3,
        backoff_factor=0.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["HEAD", "GET", "OPTIONS"]
    )
    s.mount("http://", HTTPAdapter(max_retries=retries))
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    })
    return s

_REQ = _build_requests_session()

def load_image_resized(inp, max_edge=MAX_IMAGE_EDGE, patch=14):
    """URL 이미지를 로드 → 리사이즈 → 디스크 캐시 저장/재사용 (세션/재시도/UA 사용)"""
    if isinstance(inp, str) and inp.startswith(("http://", "https://")):
        path = _cache_path(inp, max_edge)
        if os.path.exists(path):
            try:
                return Image.open(path).convert("RGB")
            except Exception:
                pass
        try:
            resp = _REQ.get(inp, timeout=(5, 10))  # (connect, read)
            resp.raise_for_status()
            img = Image.open(io.BytesIO(resp.content)).convert("RGB")
        except Exception:
            if os.path.exists(path):
                try:
                    return Image.open(path).convert("RGB")
                except Exception:
                    pass
            return None

        w, h = img.size
        if max(w, h) > max_edge:
            scale = max_edge / float(max(w, h))
            new_w = _round_to_multiple(int(w * scale), patch)
            new_h = _round_to_multiple(int(h * scale), patch)
            new_w = max(patch, new_w); new_h = max(patch, new_h)
            img = img.resize((new_w, new_h), Image.BICUBIC)
        else:
            img = img.resize((_round_to_multiple(w, patch), _round_to_multiple(h, patch)), Image.BICUBIC)
        try:
            img.save(path, format="JPEG", quality=90)
        except Exception:
            pass
        return img
    return None

def stratified_or_random_sample(df, n=8000, by_cols=None, seed=42):
    n = min(n, len(df))
    if n <= 0: return df.head(0).copy()
    if by_cols:
        tot = len(df)
        parts, remain = [], n
        for _, g in df.groupby(by_cols, dropna=False, group_keys=False):
            alloc = max(1, int(round(len(g) / tot * n)))
            take = min(alloc, len(g))
            parts.append(g.sample(n=take, random_state=seed))
            remain -= take
        out = pd.concat(parts) if parts else df.head(0)
        if remain > 0:
            rest = df.drop(out.index, errors="ignore")
            if len(rest) > 0:
                out = pd.concat([out, rest.sample(n=min(remain, len(rest)), random_state=seed)])
        if len(out) > n:
            out = out.sample(n=n, random_state=seed)
        return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    else:
        return df.sample(n=n, random_state=seed).reset_index(drop=True)

def torch_version_tuple():
    ver = torch.__version__.split('+')[0]
    return tuple(int(x) for x in ver.split('.')[:3])

# -----------------------------
# 모델/프로세서 로드 (QLoRA 4bit)
# -----------------------------
from transformers import BitsAndBytesConfig
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
)

device_map = {"": f"cuda:{torch.cuda.current_device()}"} if torch.cuda.is_available() else "cpu"
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    device_map=device_map,
    quantization_config=bnb,
    trust_remote_code=True,
    token=token
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, token=token, use_fast=True)

# k-bit 학습 준비
model = prepare_model_for_kbit_training(model)

# 비전 타워 동결
for n, p in model.named_parameters():
    if any(k in n for k in ["vision", "vision_tower", "clip", "image", "vision_model"]):
        p.requires_grad = False

# LoRA 부착
lora = LoraConfig(
    r=LORA_R, lora_alpha=16, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora)

# 캐시 끄기 (gradient checkpointing 미사용과 호환)
if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

# trainable % 출력
def count_trainable(m):
    t, a = 0, 0
    for p in m.parameters():
        a += p.numel()
        if p.requires_grad: t += p.numel()
    return t, a
t, a = count_trainable(model)
print(f"trainable params: {t:,} || all params: {a:,} || trainable%: {t/a*100:.4f}")
print("모델 + LoRA 준비 완료")

# -----------------------------
# 데이터셋
# -----------------------------
SYSTEM_INST = "You are a concise, honest, multimodal assistant."

class MultiModalDataset(TorchDataset):
    def __init__(self, df, processor):
        self.df = df.reset_index(drop=True)
        self.processor = processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        input_type = row.get("input_type", "")
        question   = row.get("question", "")
        context    = row.get("input", "")
        answer     = row.get("output", "")

        user_text = build_text_prompt(context if input_type=="text" else "", question)
        image = load_image_resized(context) if input_type == "image" else None

        if image is not None:
            messages = [
                {"role":"system", "content":[{"type":"text","text": SYSTEM_INST}]},
                {"role":"user",   "content":[{"type":"image"}, {"type":"text","text": user_text or ""}]}
            ]
            images = [image]
        else:
            messages = [
                {"role":"system", "content":[{"type":"text","text": SYSTEM_INST}]},
                {"role":"user",   "content":[{"type":"text","text": user_text or ""}]}
            ]
            images = None

        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        enc = self.processor(text=prompt, images=images, return_tensors="pt", padding=False)

        # ---------- PATCH②: 프롬프트 보존, '정답'만 컷 ----------
        enc_ids = enc["input_ids"].squeeze(0)
        enc_att = enc["attention_mask"].squeeze(0)
        ans_ids = self.processor.tokenizer(
            str(answer) if isinstance(answer, str) else "",
            return_tensors="pt", add_special_tokens=True, padding=False
        ).input_ids.squeeze(0)

        allow = max(1, MAX_TEXT_TOKENS - enc_ids.size(0))  # 정답 허용 토큰 수
        if ans_ids.size(0) > allow:
            ans_ids = ans_ids[:allow]  # 정답만 컷

        input_ids = torch.cat([enc_ids, ans_ids], dim=0)
        attention = torch.cat([enc_att, torch.ones_like(ans_ids)], dim=0)

        labels = input_ids.clone()
        labels[:enc_ids.size(0)] = -100  # 프롬프트는 라벨 제외

        item = {
            "input_ids": input_ids.to(torch.long),
            "attention_mask": attention.to(torch.long),
            "labels": labels.to(torch.long),
        }
        if "pixel_values" in enc:
            item["pixel_values"] = enc["pixel_values"].squeeze(0)
        if "image_grid_thw" in enc:
            item["image_grid_thw"] = enc["image_grid_thw"].squeeze(0)
        return item

def collate(batch):
    pad_id = processor.tokenizer.pad_token_id or 0
    def pad_1d(tensors, pad_val, dtype=torch.long):
        m = max(t.size(0) for t in tensors)
        out = []
        for t in tensors:
            if t.size(0) < m:
                pad = torch.full((m - t.size(0),), pad_val, dtype=t.dtype)
                out.append(torch.cat([t, pad], dim=0))
            else:
                out.append(t)
        return torch.stack(out, dim=0).to(dtype)
    b = {}
    b["input_ids"]      = pad_1d([x["input_ids"] for x in batch], pad_id, torch.long)
    b["attention_mask"] = pad_1d([x["attention_mask"] for x in batch], 0, torch.long)
    b["labels"]         = pad_1d([x["labels"] for x in batch], -100, torch.long)
    if all("pixel_values" in x for x in batch):
        pvs = [x["pixel_values"] for x in batch]
        if all(tuple(p.shape)==tuple(pvs[0].shape) for p in pvs):
            b["pixel_values"] = torch.stack(pvs, dim=0)
        else:
            max_h = max(p.shape[1] for p in pvs)
            max_w = max(p.shape[2] for p in pvs)
            padded = []
            for p in pvs:
                pad_h = max_h - p.shape[1]
                pad_w = max_w - p.shape[2]
                padded.append(torch.nn.functional.pad(p, (0, pad_w, 0, pad_h)))
            b["pixel_values"] = torch.stack(padded, dim=0)
    if all("image_grid_thw" in x for x in batch):
        grids = [x["image_grid_thw"] for x in batch]
        try: b["image_grid_thw"] = torch.stack(grids, dim=0)
        except Exception: b["image_grid_thw"] = grids
    return b

# -----------------------------
# 데이터 로드 (+ 8k 서브셋 고정)
# -----------------------------
df_full = safe_read_df(CSV_TRAIN).copy()
df_va   = safe_read_df(CSV_VAL).copy()
df_full["_orig_idx"] = np.arange(len(df_full))

if USE_TRAIN_SUBSET:
    os.makedirs(SAVE_DIR, exist_ok=True)
    if os.path.exists(SUBSET_IDX_PATH):
        try:
            idx_df = pd.read_csv(SUBSET_IDX_PATH)
            keep_idx = idx_df["_orig_idx"].astype(int).tolist()
            df_tr = df_full.iloc[keep_idx].reset_index(drop=True)
            print(f"★ 부분 학습 모드(재개): subset size = {len(df_tr)} (from {len(df_full)})")
        except Exception as e:
            print("부분 인덱스 로드 실패, 새로 샘플링:", e)
            by_cols = [c for c in SUBSET_GROUP_CANDIDATES if c in df_full.columns]
            df_tr = stratified_or_random_sample(df_full, n=SUBSET_SIZE, by_cols=by_cols, seed=SUBSET_SEED)
            pd.DataFrame({"_orig_idx": df_tr["_orig_idx"].tolist()}).to_csv(SUBSET_IDX_PATH, index=False)
            print(f"★ 부분 학습 모드(신규): subset size = {len(df_tr)} (from {len(df_full)}) → 인덱스 저장")
    else:
        by_cols = [c for c in SUBSET_GROUP_CANDIDATES if c in df_full.columns]
        df_tr = stratified_or_random_sample(df_full, n=SUBSET_SIZE, by_cols=by_cols, seed=SUBSET_SEED)
        pd.DataFrame({"_orig_idx": df_tr["_orig_idx"].tolist()}).to_csv(SUBSET_IDX_PATH, index=False)
        print(f"★ 부분 학습 모드(신규): subset size = {len(df_tr)} (from {len(df_full)}) → 인덱스 저장")
else:
    df_tr = df_full.reset_index(drop=True)
    print(f"train size: {len(df_tr)}")
print(f"val size: {len(df_va)}")

if "_orig_idx" in df_tr.columns:
    df_tr = df_tr.drop(columns=["_orig_idx"])

train_ds = MultiModalDataset(df_tr, processor)
val_ds   = MultiModalDataset(df_va, processor)

# -----------------------------
# 시간 제한 + 주기 저장 콜백
# -----------------------------
class TimeLimitCallback(TrainerCallback):
    def __init__(self, max_seconds, autosave_every_sec=AUTOSAVE_EVERY_MIN*60):
        self.start = time.time()
        self.max_seconds = max_seconds
        self.autosave_every_sec = autosave_every_sec
        self.last_save = self.start
    def on_step_end(self, args, state, control, **kwargs):
        now = time.time()
        if now - self.last_save >= self.autosave_every_sec:
            control.should_save = True
            self.last_save = now
        if now - self.start >= self.max_seconds:
            print("\n⏰ 시간 제한 도달: 체크포인트 저장 후 학습 종료합니다.")
            control.should_save = True
            control.should_training_stop = True
        return control

# -----------------------------
# TrainingArguments (버전 호환)
# -----------------------------
def _supports_arg(cls, name: str) -> bool:
    try: return name in inspect.signature(cls.__init__).parameters
    except Exception: return False

os.makedirs(CKPT_DIR, exist_ok=True)

# 옵티마이저 선택
try:
    torch.optim.AdamW([torch.zeros(1, requires_grad=True)])
    optim_name = "adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch"
except Exception:
    optim_name = "paged_adamw_8bit"

args_kwargs = dict(
    output_dir=CKPT_DIR,
    per_device_train_batch_size=PER_DEV_BATCH,
    per_device_eval_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=1e-4,
    num_train_epochs=1,              # 8k 러프런 기준. 전체 학습 시 늘려도 됨
    bf16=USE_BF16,
    fp16=USE_FP16,
    logging_strategy="steps",
    logging_steps=20,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim=optim_name,
    gradient_checkpointing=False,    # 안전 모드
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    save_safetensors=True,
    seed=42,
    max_grad_norm=1.0,
)
if _supports_arg(TrainingArguments, "tf32"):
    args_kwargs["tf32"] = True

# 평가 전략(epoch/steps) 옵션 호환
if _supports_arg(TrainingArguments, "evaluation_strategy"):
    args_kwargs["evaluation_strategy"] = "no"
elif _supports_arg(TrainingArguments, "eval_strategy"):
    args_kwargs["eval_strategy"] = "no"

args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=None,
    data_collator=collate,
    callbacks=[TimeLimitCallback(max_seconds=int(LIMIT_HOURS*3600))],
)

# -----------------------------
# Torch<2.6 CVE 우회: weights-only resume
# -----------------------------
def try_load_adapter_weights_from(path):
    st_path = os.path.join(path, "adapter_model.safetensors")
    if not os.path.exists(st_path):
        return False
    try:
        sd = safe_load_file(st_path)
        missing, unexpected = model.load_state_dict(sd, strict=False)
        print(f"weights-only resume: missing={len(missing)}, unexpected={len(unexpected)}")
        return True
    except Exception as e:
        print("weights-only load 실패:", e)
        return False

last_ckpt = None
if not FORCE_FRESH:
    maybe_ckpt = get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
    can_secure_resume = torch_version_tuple() >= (2,6,0)
    if maybe_ckpt and not can_secure_resume:
        print(f"🔁 Torch<{2.6}: optimizer 복원은 생략하고 가중치만 이어서 학습합니다. ({maybe_ckpt})")
        ok = try_load_adapter_weights_from(maybe_ckpt)
        if not ok and os.path.isdir(FINAL_DIR):
            print("체크포인트 로드 실패 → FINAL_DIR에서 가중치 복원 시도")
            ok = try_load_adapter_weights_from(FINAL_DIR)
        last_ckpt = None  # resume_from_checkpoint는 사용하지 않음
    else:
        last_ckpt = maybe_ckpt
        if last_ckpt:
            print(f"🔁 체크포인트에서 재개(정상): {last_ckpt}")
        else:
            print("🚀 새 학습 시작")
else:
    print("🚀 FORCE_FRESH=True → 새 학습 시작(이전 가중치 무시)")

# -----------------------------
# 학습 & 저장
# -----------------------------
try:
    trainer.train(resume_from_checkpoint=last_ckpt)
except Exception as e:
    print(f"⚠️ 학습 중 예외 발생: {e}\n현재 상태 저장을 시도합니다.")
finally:
    os.makedirs(FINAL_DIR, exist_ok=True)
    trainer.save_model(FINAL_DIR)
    processor.save_pretrained(FINAL_DIR)
    print("모델/프로세서 저장 완료 →", FINAL_DIR)
    torch.cuda.empty_cache(); gc.collect()

# -----------------------------
# 추론 → submission.csv (부분 저장/재개 + 오토그로우)
# -----------------------------
print("\n=== inference on test set (autogrow) ===")
model.eval(); torch.set_grad_enabled(False)
df_test = safe_read_df(CSV_TEST).reset_index(drop=True)

def build_messages_for_infer(row):
    input_type = row.get("input_type", "")
    question   = row.get("question", "")
    context    = row.get("input", "")
    task       = str(row.get("task", "")).lower()
    user_text  = build_text_prompt(context if input_type=="text" else "", question)
    image      = load_image_resized(context)

    if image is not None and input_type == "image":
        messages = [
            {"role":"system", "content":[{"type":"text","text": SYSTEM_INST}]},
            {"role":"user",   "content":[{"type":"image"}, {"type":"text","text": user_text or ""}]}
        ]
        images = [image]
    else:
        messages = [
            {"role":"system", "content":[{"type":"text","text": SYSTEM_INST}]},
            {"role":"user",   "content":[{"type":"text","text": user_text or ""}]}
        ]
        images = None
    return messages, images, question, task, input_type

# ---- EOS / PAD 명시 ----
EOS_IDS = []
tok = processor.tokenizer
if getattr(tok, "eos_token_id", None) is not None:
    EOS_IDS.append(tok.eos_token_id)
for tkn in ["<|im_end|>", "</s>", "<|endoftext|>"]:
    try:
        tid = tok.convert_tokens_to_ids(tkn)
        if isinstance(tid, int) and tid >= 0:
            EOS_IDS.append(tid)
    except Exception:
        pass
EOS_IDS = sorted(set([i for i in EOS_IDS if i is not None]))
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else (EOS_IDS[0] if EOS_IDS else None)

def _is_complete(text: str) -> bool:
    s = (text or "").strip()
    if not s: return False
    return bool(re.search(r'[.!?」”’\'")\]\}]$', s))

DECODE_PROFILE = {
    "vqa":     dict(init=64,  step=32,  limit=128,  min_new=16),
    "ocr":     dict(init=160, step=80,  limit=320,  min_new=24),
    "caption": dict(init=192, step=96,  limit=384,  min_new=24),
    "docqa":   dict(init=224, step=96,  limit=448,  min_new=24),
    "other":   dict(init=128, step=64,  limit=256,  min_new=16),
}
def _prof(task: str):
    t = (task or "").lower()
    if t in DECODE_PROFILE: return DECODE_PROFILE[t]
    if "caption" in t: return DECODE_PROFILE["caption"]
    if "ocr" in t or "read" in t: return DECODE_PROFILE["ocr"]
    if "doc" in t or "long" in t: return DECODE_PROFILE["docqa"]
    if "vqa" in t or "qa" in t:   return DECODE_PROFILE["vqa"]
    return DECODE_PROFILE["other"]

def smart_generate_autogrow(base_enc, task):
    """멀티모달 컨텍스트를 유지한 채 상한을 키워가며 이어 생성."""
    prof = _prof(task)
    # 공통 생성 옵션(그리디 + 반복 억제)
    base_cfg = dict(
        do_sample=False,
        num_beams=1,
        no_repeat_ngram_size=4,
        repetition_penalty=1.05,
        pad_token_id=PAD_ID,
    )
    if EOS_IDS:
        base_cfg["eos_token_id"] = EOS_IDS if len(EOS_IDS) > 1 else EOS_IDS[0]

    # 첫 입력(컨텍스트 + 이미지)
    enc = {k:v for k,v in base_enc.items()}
    if torch.cuda.is_available():
        enc = {k:(v.to(model.device) if hasattr(v, "to") else v) for k,v in enc.items()}

    cur_cap = prof["init"]
    step    = prof["step"]
    limit   = prof["limit"]
    min_new = prof["min_new"]

    out = model.generate(**enc, max_new_tokens=cur_cap, min_new_tokens=min_new, **base_cfg)
    prefix_len = base_enc["input_ids"].shape[-1]
    new_ids = out[0][prefix_len:]
    text = tok.decode(new_ids, skip_special_tokens=True).strip()

    tries = 0
    while (tries < 3) and (len(new_ids) >= cur_cap - 4) and (not _is_complete(text)) and (cur_cap < limit):
        tries += 1
        cur_cap = min(cur_cap + step, limit)

        # 이어쓰기 입력: (원래 프롬프트 + 지금까지의 생성) + 같은 pixel_values/grid
        new_input_ids = torch.cat([base_enc["input_ids"][0].to(out.device), new_ids.to(out.device)], dim=0).unsqueeze(0)
        new_attention = torch.ones_like(new_input_ids)

        next_inp = {
            "input_ids": new_input_ids,
            "attention_mask": new_attention,
        }
        if "pixel_values" in base_enc:     next_inp["pixel_values"]     = base_enc["pixel_values"].to(out.device)
        if "image_grid_thw" in base_enc:   next_inp["image_grid_thw"]   = base_enc["image_grid_thw"].to(out.device)

        out = model.generate(**next_inp, max_new_tokens=step, min_new_tokens=min_new, **base_cfg)
        new_ids = out[0][prefix_len:]
        text = tok.decode(new_ids, skip_special_tokens=True).strip()

    return text

def clean_output(text, task, question=""):
    s = (text or "").strip()
    m = re.search(r'####\s*([-\d][\d,\.]*)', s)  # GSM8K 스타일
    if m: return m.group(1).replace(",", "")
    if (task or "").lower() == "vqa" and "how many" in (question or "").lower():
        m = re.search(r'-?\d+', s.replace(",", ""))
        if m: return m.group(0)
    return s

# 부분 저장 복구
processed = {}
if os.path.exists(SUBMIT_PART):
    try:
        dfp = pd.read_csv(SUBMIT_PART)
        for _, r in dfp.iterrows():
            processed[int(r["id"])] = str(r["output"])
        print(f"🔁 기존 부분 결과 로드: {len(processed)}개")
    except Exception as e:
        print("부분 결과 로드 실패, 새로 시작:", e)

preds_map = dict(processed)

# 남은 시간만 사용
elapsed_total = time.time() - GLOBAL_START
limit_sec = max(0, int(LIMIT_HOURS*3600) - int(elapsed_total))
infer_start = time.time()
def time_left_ok():
    return (time.time() - infer_start) < max(0, limit_sec - 5*60)

todo_ids = [i for i in range(len(df_test)) if i not in preds_map]
for idx, i in enumerate(todo_ids, 1):
    if not time_left_ok():
        print("⏰ 추론 시간 한계 도달. 부분 결과 저장 후 종료.")
        break
    row = df_test.iloc[i]
    try:
        messages, images, qtxt, task, itype = build_messages_for_infer(row)
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        base_enc = processor(text=prompt, images=images, return_tensors="pt", padding=False)
        # 오토그로우 생성
        raw = smart_generate_autogrow(base_enc, task)
        pred = clean_output(raw, task, qtxt)
    except Exception as e:
        print(f"infer error @ {i}:", e)
        pred = ""

    preds_map[i] = pred

    if (idx % INFER_SAVE_EVERY == 0) or (idx == len(todo_ids)):
        tmp = pd.DataFrame({"id": sorted(preds_map.keys()),
                            "output": [preds_map[k] for k in sorted(preds_map.keys())]})
        tmp.to_csv(SUBMIT_PART, index=False, encoding="utf-8")
        print(f"[{idx}/{len(todo_ids)}] 부분 저장 → {SUBMIT_PART}")

# 최종 병합 저장
all_ids = list(range(len(df_test)))
outputs = [preds_map.get(i, "") for i in all_ids]
pd.DataFrame({"id": all_ids, "output": outputs}).to_csv(SUBMIT_OUT, index=False, encoding="utf-8")
print("\n" + "="*60)
print("✅ submission.csv saved to:", SUBMIT_OUT)
print("rows:", len(all_ids), "| empty:", sum(1 for x in outputs if (x or "")==""))
print("="*60)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


torch: 2.3.1+cu121 | CUDA: 12.1


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
20,1.367000
40,1.035000
60,0.776900
80,0.720000
100,0.660900
120,0.656100
140,0.637600
160,0.601300
180,0.649300
200,0.634200


모델/프로세서 저장 완료 → /content/drive/MyDrive/multimodal_challenge_aju/outputs_qwen2vl_lora_total_last_last4

=== inference on test set (autogrow) ===


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🔁 기존 부분 결과 로드: 1400개
[200/1093] 부분 저장 → /content/drive/MyDrive/multimodal_challenge_aju/outputs_qwen2vl_lora_total_last_last4/submission.csv.partial
[400/1093] 부분 저장 → /content/drive/MyDrive/multimodal_challenge_aju/outputs_qwen2vl_lora_total_last_last4/submission.csv.partial
